# External outputs in object storage

Run the installed `FileGenerator` on a Modal worker and verify its accepted
external files in an S3-compatible store. The controller publishes managed
files only after validating the selected worker result. This replaces the
old execute-only output archive interface; there is no per-call `output_store`
or presigned archive to treat as an accepted artifact.

This optional tutorial makes billable Modal and object-storage calls. Install
matching Artisan and `artisan-modal` packages, configure Modal authentication,
and supply a canonical Pixi manifest and lock containing both exact packages.
Build a checkout wheel with `python -m pip wheel --no-deps .` in an isolated
build frontend; the remote project also needs the matching provider wheel.
Do not point a remote worker at an editable notebook directory.

Set `ARTISAN_MODAL_WORKER_MANIFEST` to that project's absolute manifest path.
Its `default` environment must contain the matching packages. Each notebook
uses a unique application name so preparation does not replace another run.
The environment cache follows `XDG_CACHE_HOME` (or `~/.cache`); set that standard
variable to an absolute writable directory when you need an isolated cache.
Set `ARTISAN_S3_BUCKET`, `ARTISAN_S3_ENDPOINT_URL`, `AWS_ACCESS_KEY_ID`, and
`AWS_SECRET_ACCESS_KEY` to a dedicated writable tutorial bucket/prefix. The
credentials are explicit runtime storage configuration for the controller and
worker; they do not belong in a wheel, Pixi lock, or scientific parameters.
The example uses an HTTPS endpoint. Use a narrowly scoped credential and keep
it out of notebook output. Missing setup raises an error before preparation.

Preparation is explicit and may build images and deploy named functions. Merely
constructing or selecting a runner does not provision software. Provider startup,
packing, and cancellation require qualification against the actual deployed
version; local examples alone do not establish those properties.

In [ ]:
from __future__ import annotations

import os
import shutil
from pathlib import Path
from tempfile import mkdtemp
from uuid import uuid4

from artisan_modal import ModalRunner

from artisan.schemas import Environment, StorageConfig

# This must be a locked project containing matching Artisan and provider wheels.
manifest = (
    Path(os.environ["ARTISAN_MODAL_WORKER_MANIFEST"]).expanduser().resolve(strict=True)
)
bucket = os.environ["ARTISAN_S3_BUCKET"]
endpoint = os.environ["ARTISAN_S3_ENDPOINT_URL"]
key = os.environ["AWS_ACCESS_KEY_ID"]
secret = os.environ["AWS_SECRET_ACCESS_KEY"]
region = os.environ.get("AWS_DEFAULT_REGION", "auto")
storage = StorageConfig(
    protocol="s3",
    options={
        "key": key,
        "secret": secret,
        "client_kwargs": {"endpoint_url": endpoint, "region_name": region},
    },
    delta_options={
        "AWS_ACCESS_KEY_ID": key,
        "AWS_SECRET_ACCESS_KEY": secret,
        "AWS_ENDPOINT_URL": endpoint,
        "AWS_REGION": region,
    },
)
workspace = Path(mkdtemp(prefix="artisan-modal-tutorial-"))
cache_root = (
    Path(os.environ.get("XDG_CACHE_HOME", Path.home() / ".cache"))
    / "artisan/environments"
)
app_name = f"artisan-tutorial-{uuid4().hex[:12]}"
prefix = f"s3://{bucket}/artisan-tutorial/{uuid4().hex}"
delta_root, staging_root, files_root = (
    f"{prefix}/{name}" for name in ("delta", "staging", "files")
)
runner = ModalRunner(
    environment=Environment.from_pixi(manifest),
    app_name=app_name,
    cache_root=cache_root,
)
fs = storage.filesystem()
read_options = {
    "fs": fs,
    "storage_options": storage.delta_storage_options(),
    "files_root": files_root,
}

In [ ]:
from artisan.inspection import inspect_evidence
from artisan.operations.examples import FileGenerator
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.schemas import FileRefArtifact, RunnerResources
from artisan.storage import ArtifactStore

resources = RunnerResources(cpus=1, memory_gb=4, time_limit="00:05:00")
parameters = {"count": 2, "file_size_bytes": 4096, "seed": 42}
runner.prepare(FileGenerator, params=parameters, runner_resources=resources)

## Produce and accept two files

The remote worker writes to owned scratch, seals its result, and exposes only
that selected result for acceptance. Managed output files, table metadata, and
staging each have their own explicit roots under our unique prefix.

In [ ]:
with PipelineManager.create(
    name="modal_files",
    delta_root=delta_root,
    staging_root=staging_root,
    files_root=files_root,
    working_root=str(workspace),
    storage=storage,
    default_step_runner=runner,
) as pipeline:
    result = pipeline.run(
        FileGenerator, params=parameters, runner_resources=resources, skip_cache=True
    )
    summary = pipeline.finalize()
assert summary["overall_success"]
assert result.disposition is StepDisposition.EXECUTED

## Read the accepted content

Query current-run membership before hydrating the selected IDs. Hydration checks
the complete managed file identity. Listing arbitrary bucket objects would not
establish that they belong to this result.

In [ ]:
references = inspect_evidence(
    "query_artifacts",
    {
        "pipeline_run_id": pipeline.config.pipeline_run_id,
        "artifact_type": "file_ref",
        "limit": 100,
    },
    delta_root=delta_root,
    files_root=files_root,
    storage=storage,
)
selected = [
    item for item in references.items if item.current_step_number == result.step_number
]
assert len(selected) == 2
store = ArtifactStore(delta_root, **read_options)
for reference in selected:
    artifact = store.get_artifact(reference.artifact_id)
    assert isinstance(artifact, FileRefArtifact)
    assert artifact.size_bytes == 4096
    assert artifact.path.startswith(files_root + "/")
    with fs.open(artifact.path, "rb") as stream:
        assert len(stream.read()) == 4096
    print(reference.artifact_id, artifact.size_bytes)

## Cleanup

Run cleanup only after successful finalization and inspection. It closes the
pipeline again before deleting this notebook's unique store prefix and local
working directory. A failed or unresolved run keeps both for diagnosis. The
remote deployment and prepared environment remain separate resources; remove
the application created for `app_name` through Modal after you finish. Production
retention should preserve complete store evidence and its managed files together.

In [ ]:
pipeline.close()
assert summary["overall_success"]
if fs.exists(prefix):
    fs.rm(prefix, recursive=True)
shutil.rmtree(workspace)

See [Configure S3-Compatible Storage](../../how-to-guides/configuring-s3.md)
for credential and endpoint settings, and
[Run Remote Workers](../../how-to-guides/running-remote-workers.md) for deployment
and cancellation. These optional cells require actual provider qualification;
the local tutorial test suite does not exercise cloud behavior.